In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import json
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/medrag"
DATA_PATH = f"{DRIVE_BASE}/pubmedqa_filtered.json"
CHECKPOINT_DIR = f"{DRIVE_BASE}/biomistral_lens_checkpoints"
OUTPUT_DIR = f"{DRIVE_BASE}/biomistral_trial4_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Corpus loaded: {len(corpus)} samples")

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Corpus loaded: 759 samples


In [3]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)

model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size

print(f"Model loaded: {n_layers} layers, hidden size {hidden_size}, vocab {vocab_size}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded: 32 layers, hidden size 4096, vocab 32000


In [4]:
class TunedLensTranslator(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.translator = nn.Linear(hidden_size, hidden_size, bias=True)
        nn.init.eye_(self.translator.weight)
        nn.init.zeros_(self.translator.bias)

    def forward(self, hidden_state):
        return self.translator(hidden_state.float())

translators = nn.ModuleList([
    TunedLensTranslator(hidden_size) for _ in range(n_layers)
])

for layer_idx in range(n_layers):
    ckpt = torch.load(
        os.path.join(CHECKPOINT_DIR, f"translator_layer_{layer_idx:02d}.pt"),
        map_location="cuda"
    )
    translators[layer_idx].load_state_dict(ckpt["state_dict"])

translators = translators.to("cuda")
translators.eval()

for param in translators.parameters():
    param.requires_grad = False

print(f"Tuned-lens loaded: {n_layers} translators ready")

Tuned-lens loaded: 32 translators ready


In [6]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready. {index.ntotal} documents indexed.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Retriever ready. 2542 documents indexed.


In [7]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(scores[i]), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(s), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}

    for rank, r in enumerate(bm25_results):
        combined[r["abstract"]] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    return [v["meta"] for v in sorted(combined.values(),
            key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return prompt, results

def build_suppressed_prompt(query):
    return f"Question: {query}\nAnswer:"

print("Retrieval and prompt functions ready.")

Retrieval and prompt functions ready.


In [8]:
RAW_LENS_LAYERS = set()

def project_hidden_with_tuned_lens(hidden_state, layer_idx):
    with torch.no_grad():
        if layer_idx in RAW_LENS_LAYERS:
            normed = model.model.norm(hidden_state.half())
            logits = model.lm_head(normed).float()
        else:
            translated = translators[layer_idx](hidden_state.float())
            normed = model.model.norm(translated.half())
            logits = model.lm_head(normed).float()

        probs = torch.softmax(logits, dim=-1)

    return probs

def kl_between_probs(p, q):
    kl = torch.sum(p * torch.log((p + 1e-10) / (q + 1e-10)), dim=-1)
    return kl

print("Projection and KL functions ready.")
print(f"Raw logit lens used at layers: {sorted(RAW_LENS_LAYERS)}")

Projection and KL functions ready.
Raw logit lens used at layers: []


In [9]:
hook_storage = {"hidden_states": {}}
hooks = []

def make_hook(layer_idx):
    def hook(module, input, output):
        hook_storage["hidden_states"][layer_idx] = output[0].detach().squeeze(0)
    return hook

for i, block in enumerate(model.model.layers):
    hooks.append(block.register_forward_hook(make_hook(i)))

def run_hidden_pass(prompt, max_length=512):
    hook_storage["hidden_states"].clear()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to("cuda")

    seq_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_by_layer = {
        layer_idx: hook_storage["hidden_states"][layer_idx][-1, :].unsqueeze(0)
        for layer_idx in range(n_layers)
    }

    return hidden_by_layer, seq_len

print(f"Hooks registered on {n_layers} layers.")
print("Hidden pass function ready.")

Hooks registered on 32 layers.
Hidden pass function ready.


In [10]:
N_SAMPLES = len(corpus)
trial4_results = []

print(f"Running Trial 4 on {N_SAMPLES} samples...\n")

for idx, sample in enumerate(tqdm(corpus[:N_SAMPLES])):
    query = sample["query"]

    rag_prompt, retrieved = build_rag_prompt(query, k=3)
    suppressed_prompt = build_suppressed_prompt(query)

    rag_hidden, rag_seq_len = run_hidden_pass(rag_prompt)
    sup_hidden, sup_seq_len = run_hidden_pass(suppressed_prompt)

    kl_trajectory = []

    for layer_idx in range(n_layers):
        rag_probs = project_hidden_with_tuned_lens(rag_hidden[layer_idx], layer_idx)
        sup_probs = project_hidden_with_tuned_lens(sup_hidden[layer_idx], layer_idx)

        kl = kl_between_probs(rag_probs, sup_probs)
        kl_trajectory.append(float(kl[0]))

    trial4_results.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "label": sample["label"],
        "rag_seq_len": rag_seq_len,
        "sup_seq_len": sup_seq_len,
        "retrieved_pubids": [r["pubid"] for r in retrieved],
        "kl_trajectory": kl_trajectory,
        "mean_kl": float(np.mean(kl_trajectory[1:])),
        "max_kl_layer": int(np.argmax(kl_trajectory[1:]) + 1),
        "max_kl_value": float(np.max(kl_trajectory[1:])),
        "final_layer_kl": kl_trajectory[-1]
    })

    if (idx + 1) % 50 == 0:
        ckpt_path = os.path.join(OUTPUT_DIR, f"trial4_checkpoint_{idx+1}.json")
        with open(ckpt_path, "w") as f:
            json.dump(trial4_results, f)
        torch.cuda.empty_cache()
        tqdm.write(f"Checkpoint saved at sample {idx+1}")

print(f"\nTrial 4 complete. {len(trial4_results)} samples processed.")

Running Trial 4 on 759 samples...



  7%|▋         | 51/759 [00:08<01:51,  6.32it/s]

Checkpoint saved at sample 50


 13%|█▎        | 101/759 [00:16<01:39,  6.62it/s]

Checkpoint saved at sample 100


...

Checkpoint saved at sample 700


 99%|█████████▉| 751/759 [01:55<00:01,  6.15it/s]

Checkpoint saved at sample 750


100%|██████████| 759/759 [01:56<00:00,  6.50it/s]


Trial 4 complete. 759 samples processed.


In [11]:
print("TRIAL 4 SUMMARY BY LABEL \n")

for label in ["yes", "no"]:
    label_samples = [r for r in trial4_results if r["label"] == label]
    print(f"Label: {label} | n={len(label_samples)}")
    print(f"  Mean KL RAG vs suppressed: {np.mean([r['mean_kl'] for r in label_samples]):.4f}")
    print(f"  Mean peak layer:           {np.mean([r['max_kl_layer'] for r in label_samples]):.1f}")
    print(f"  Mean final layer KL:       {np.mean([r['final_layer_kl'] for r in label_samples]):.4f}")
    print()

output_path = os.path.join(OUTPUT_DIR, "trial4_full_results.json")

with open(output_path, "w") as f:
    json.dump({
        "model_id": MODEL_ID,
        "n_samples": len(trial4_results),
        "n_layers": n_layers,
        "vocab_size": vocab_size,
        "raw_lens_layers": sorted(RAW_LENS_LAYERS),
        "summary_by_label": {
            label: {
                "n": sum(1 for r in trial4_results if r["label"] == label),
                "mean_kl": float(np.mean([r["mean_kl"] for r in trial4_results
                                         if r["label"] == label])),
                "mean_peak_layer": float(np.mean([r["max_kl_layer"] for r in trial4_results
                                                  if r["label"] == label])),
                "mean_final_kl": float(np.mean([r["final_layer_kl"] for r in trial4_results
                                                if r["label"] == label]))
            } for label in ["yes", "no"]
        },
        "samples": trial4_results
    }, f, indent=2)

for fname in os.listdir(OUTPUT_DIR):
    if fname.startswith("trial4_checkpoint"):
        os.remove(os.path.join(OUTPUT_DIR, fname))

for h in hooks:
    h.remove()

print(f"Results saved to {output_path}")
print("05_biomistral_trial4_kl_rag_suppressed complete")

TRIAL 4 SUMMARY BY LABEL 

Label: yes | n=469
  Mean KL RAG vs suppressed: 0.5805
  Mean peak layer:           17.5
  Mean final layer KL:       0.7241

Label: no | n=290
  Mean KL RAG vs suppressed: 0.4720
  Mean peak layer:           19.0
  Mean final layer KL:       0.6539

Results saved to /content/drive/MyDrive/medrag/biomistral_trial4_outputs/trial4_full_results.json
05_biomistral_trial4_kl_rag_suppressed complete
